In [1]:
import os
from openai import OpenAI

In [2]:
PROMPT_FILE = "prompt/conversation-generator.md"

In [3]:
if not os.path.exists(PROMPT_FILE):
    raise FileNotFoundError(f"Could not find {PROMPT_FILE}! Ensure it is in the same folder as this script.")

with open(PROMPT_FILE, "r", encoding="utf-8") as f:
    system_instructions = f.read()

In [ ]:
client = OpenAI(
  base_url = "https://integrate.api.nvidia.com/v1",
  api_key = "nvapi-xxxxxxxxxxxxxxxxxxxxxxxxxxxxx"  # Replace with your actual NVIDIA API key
)


completion = client.chat.completions.create(
  model="nvidia/nemotron-3-ultra-550b-a55b",
  messages=[{"role": "system", "content": system_instructions}],
  temperature=1,
  top_p=0.95,
  max_tokens=16384,
  extra_body={"chat_template_kwargs":{"enable_thinking":True},"reasoning_budget":16384},
  stream=True
)

for chunk in completion:
  if not chunk.choices:
    continue
  reasoning = getattr(chunk.choices[0].delta, "reasoning_content", None)
  if reasoning:
    print(reasoning, end="")
  if chunk.choices[0].delta.content is not None:
    print(chunk.choices[0].delta.content, end="")

We are to generate 20 conversations, each with 5 user turns (so 5 assistant responses, but the last assistant response is the answer we need to predict). The conversation history in "question" includes all previous turns (user and assistant) and ends with the 5th user turn. The "answer" is the function call for the 5th user turn.

We must use only the provided tools. The tools are:
- toggle_lights (room, state)
- set_thermostat (temperature, mode)
- lock_door (door, state)
- get_device_status (device_type, room)
- set_scene (scene)
- intent_unclear (no parameters)

We need to generate varied, natural conversations with emotional context, pronouns, follow-ups, etc. At least one sample must use intent_unclear. Also, we must vary rooms, devices, states, temperatures, scenes.

We have to output JSONL: each line is a JSON object with "question" and "answer", both stringified JSON.

Let's plan 20 conversations. Each conversation has 5 turns. We'll design each conversation step by step.

We n

KeyboardInterrupt: 